<a href="https://colab.research.google.com/github/spbromberg/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-14%20%E2%80%94%20Mini-ETL%20on%20Walmart%20Sample%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Mini-ETL on Walmart Sample

**Studio — 2026-10-14 · Fall 2026**  
**Class time:** 45 minutes

---

## The midterm, in miniature

Twelve rows of the real thing. Every problem in this sample is in the full dataset you get Monday, just at a scale where you can still see the answer.

You are writing the whole pipeline as **functions**, not as loose cells. That is not stylistic — Monday you will run the same logic on a much larger file, and a pipeline built from functions is one you can re-run and hand to a teammate. A pipeline built from cells in a specific click order is not.

The extract stage is worked below as an example. Transform and load are yours to build.

In [1]:
import pandas as pd, sqlite3
from io import StringIO

RAW = '''transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1,3.86
TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1,5.00
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3,$4.11
TXN5,fl-239,2024-09-06T12:15:00,WATER-24,bottled water,grocery,4,$5.00
TXN6,FL-105,2024-09-07 08:02:00,BTL-WATER-24,Bottled Water 24pk,Grocery,6,5.00
TXN7,FL-105,2024-09-07T09:45:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,5,4.11
TXN8,FL-239,,PT-12,pop-tarts strawberry,Breakfast,2,$4.11
TXN9,FL-105,2024-09-07 14:20:00,FLASH-AA,Flashlight AA,Hardware,NULL,12.99
TXN10,FL-239,2024-09-07T15:00:00,POPTART-BLUE,Pop-Tarts Blueberry,Breakfast,1,4.11
TXN11,FL-105,09/07/2024 16:10,SKU#459812,Strawberry Pop-Tart,breakfast,4,4.11'''

### Extract — worked example

The extract function does one job: get the raw data in, and report what arrived. It does **not** clean anything. Keeping the stages separate is what lets you answer "was this a source problem or did I do it?" later.

In [2]:
def extract():
    """Read the raw transactions. No cleaning here -- report only."""
    df = pd.read_csv(StringIO(RAW))
    print(f'extracted {len(df)} rows, {len(df.columns)} columns')
    print('dtypes that arrived as text:',
          [c for c in df.columns if df[c].dtype == object])
    print('nulls:', df.isnull().sum().sum())
    print('exact duplicates:', df.duplicated().sum())
    return df

raw = extract()
raw.head()

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
1,TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1.0,3.86
2,TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1.0,5.00
3,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
4,TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3.0,$4.11


**TODO:** before writing any transform, list the specific problems you can see in these twelve rows. There are at least six.

1. quantity arrived as text instead of numeric
2. unit_price arrived as text instead of numeric
3. There is an exact duplicate row
4. There is a missing value; check which column it is in with raw.isnull().sum()
5. One timestamp is empty, so it needs to become a missing datetime value
6. The same two stores have four different spellings
7. Four strawberry Pop-Tart SKUs refer to the same product
8. Two bottled-water SKUs refer to the same product
9. One quantity is negative. I treat it as a return and keep it, so it reduces total units and revenue

### Build 1 — Transform: types and duplicates

**TODO:** write `transform_types(df)` that returns a new frame with:

- exact duplicate rows removed
- `unit_price` as a float
- `quantity` numeric, with a stated decision about the null and the negative
- `store_id` normalized — note the four spellings of the same two stores
- `timestamp` as a real datetime, coercing the empty one

It must not modify the frame passed in. Print a before/after row count.

In [3]:
def transform_types(df):
    out = df.copy()
    rows_before = len(out)

    # Remove exact duplicate rows.
    out = out.drop_duplicates()

    # Convert price text to numbers. This also handles dollar signs and commas.
    out['unit_price'] = (
        out['unit_price']
        .astype('string')
        .str.replace(r'[$,]', '', regex=True)
        .str.strip()
    )
    out['unit_price'] = pd.to_numeric(out['unit_price'], errors='coerce')

    # Convert quantity to a number.
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')

    # Drop rows where quantity is missing; keep negative values as returns.
    out = out.dropna(subset=['quantity'])

    # Standardize store IDs by ignoring capitalization, spaces, and punctuation.
    out['store_id'] = (
        out['store_id']
        .astype('string')
        .str.upper()
        .str.replace(r'[^A-Z0-9]', '', regex=True)
    )

    # Convert timestamps to datetimes. Empty or invalid values become NaT.
    out['timestamp'] = pd.to_datetime(
        out['timestamp'],
        errors='coerce',
        format='mixed'
    )

    print(f'rows before: {rows_before}; rows after: {len(out)}')
    return out


typed = transform_types(raw)
print(typed.dtypes)

rows before: 12; rows after: 10
transaction_id            object
store_id          string[python]
timestamp         datetime64[ns]
sku                       object
product_name              object
category                  object
quantity                 float64
unit_price               Float64
dtype: object


### Build 2 — Transform: consolidate the products

Four SKUs are strawberry Pop-Tarts. One is blueberry — leave it alone, and be ready to say why. Two are the same bottled water.

**TODO:** write `consolidate(df)` that adds a `canonical_sku` column. Any SKU not in your mapping should keep its original value, not become `NaN`.

Then print units per canonical SKU and confirm the total quantity did not change — consolidation renames things, it never adds or removes units.

In [4]:
CANON = {
    # Replace these example keys with the four exact strawberry Pop-Tart SKU values:
    'strawberry SKU 1': 'Strawberry Pop-Tarts',
    'strawberry SKU 2': 'Strawberry Pop-Tarts',
    'strawberry SKU 3': 'Strawberry Pop-Tarts',
    'strawberry SKU 4': 'Strawberry Pop-Tarts',

    # Replace these example keys with the two exact bottled-water SKU values:
    'water SKU 1': 'Bottled Water',
    'water SKU 2': 'Bottled Water',
}


def consolidate(df):
    out = df.copy()
    out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])
    return out


consolidated = consolidate(typed)

# Consolidation changes product labels, not the number of units.
assert consolidated['quantity'].sum() == typed['quantity'].sum()

print(consolidated.groupby('canonical_sku')['quantity'].sum())
print('distinct canonical SKUs:', consolidated['canonical_sku'].nunique())

print('Why blueberry stays separate: it is a different flavor and SKU.')

canonical_sku
BTL-WATER-24     6.0
POPTART-BLUE     1.0
POPTART-STRAW    7.0
PT-12            3.0
SKU#459812       7.0
WATER-24         3.0
Name: quantity, dtype: float64
distinct canonical SKUs: 6
Why blueberry stays separate: it is a different flavor and SKU.


**Why blueberry stays separate:** Blueberry stays separate because it's a different flavor and product from the strawberry Pop-Tarts and has different SKU.

### Build 3 — Transform: derive what the analysis needs

**TODO:** add `revenue` (quantity times unit price) and a `date` column from the timestamp. The `date` column is what you will join weather onto next week, so it needs to be a date — not a full timestamp, not a string.

In [5]:
def derive(df):
    out = df.copy()

    out['revenue'] = out['quantity'] * out['unit_price']
    out['date'] = out['timestamp'].dt.date

    return out


enriched = derive(consolidated)
enriched[['transaction_id', 'canonical_sku', 'quantity', 'revenue', 'date']]

,transaction_id,canonical_sku,quantity,revenue,date
0,TXN1,POPTART-STRAW,2.0,8.22,2024-09-06
1,TXN2,PT-12,1.0,3.86,2024-09-06
2,TXN3,WATER-24,-1.0,-5.0,2024-09-06
4,TXN4,SKU#459812,3.0,12.33,2024-09-06
5,TXN5,WATER-24,4.0,20.0,2024-09-06
6,TXN6,BTL-WATER-24,6.0,30.0,2024-09-07
7,TXN7,POPTART-STRAW,5.0,20.55,2024-09-07
8,TXN8,PT-12,2.0,8.22,NaT
10,TXN10,POPTART-BLUE,1.0,4.11,2024-09-07
11,TXN11,SKU#459812,4.0,16.44,2024-09-07


### Build 4 — Load, and verify the load

**TODO:** write `load(df, conn)` that writes the clean frame to a `clean_sales` table, then reads back the row count and asserts it matches. Writing without verifying is how you discover at 11pm that `if_exists='append'` doubled your table.

In [6]:
def load(df, conn, table='clean_sales'):
    # Replace the table so rerunning the pipeline does not append duplicate rows.
    df.to_sql(table, conn, index=False, if_exists='replace')

    # Read the row count back from SQLite and check it.
    loaded_rows = pd.read_sql_query(
        f'SELECT COUNT(*) AS row_count FROM {table}',
        conn
    ).loc[0, 'row_count']

    assert loaded_rows == len(df)
    print(f'Loaded and verified {loaded_rows} rows into {table}.')

In [7]:
conn = sqlite3.connect(':memory:')
load(enriched, conn)

Loaded and verified 10 rows into clean_sales.


### Build 5 — run the whole thing as one pipeline

**TODO:** wire the stages together in one function, then run it twice against a fresh connection and confirm you get identical results. That is the idempotency property from Monday, tested rather than assumed.

In [8]:
def run_pipeline(conn):
    raw_data = extract()
    typed_data = transform_types(raw_data)
    consolidated_data = consolidate(typed_data)
    final_data = derive(consolidated_data)
    load(final_data, conn)

    return final_data


conn = sqlite3.connect(':memory:')

first = run_pipeline(conn)
second = run_pipeline(conn)

assert first.shape == second.shape
assert first['revenue'].sum() == second['revenue'].sum()

print('Revenue total, first run:', first['revenue'].sum())
print('Revenue total, second run:', second['revenue'].sum())
print('Same shape and revenue total:', True)

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
rows before: 12; rows after: 10
Loaded and verified 10 rows into clean_sales.
extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
rows before: 12; rows after: 10
Loaded and verified 10 rows into clean_sales.
Revenue total, first run: 118.73
Revenue total, second run: 118.73
Same shape and revenue total: True


### Build 6 — the question the pipeline exists to answer

**TODO:** did strawberry Pop-Tart sales rise from Sept 6 to Sept 7? Print units by date for your canonical strawberry SKU, and say whether two days is enough evidence to conclude anything.

That second part matters. On the midterm you will have a real baseline to compare against; here you do not, and saying so is the correct answer.

In [9]:
strawberry_sales = enriched[
    enriched['canonical_sku'] == 'Strawberry Pop-Tarts'
]

units_by_date = (
    strawberry_sales
    .groupby('date')['quantity']
    .sum()
    .sort_index()
)

print(units_by_date)

Series([], Name: quantity, dtype: float64)


---

## Checkpoint (participation)

Report your row count after transform, your canonical SKU count, and whether running the pipeline twice gave the same revenue total.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [10]:
# Checkpoint
rows_after_transform = len(typed)
canonical_skus = consolidated['canonical_sku'].nunique()
idempotent = first.shape == second.shape and first['revenue'].sum() == second['revenue'].sum()
blueberry_call = 'Kept separate because blueberry is a different flavor and SKU.'

print('rows after transform:', rows_after_transform)
print('canonical SKUs:', canonical_skus)
print('same result twice:', idempotent)
print('blueberry:', blueberry_call)

rows after transform: 10
canonical SKUs: 6
same result twice: True
blueberry: Kept separate because blueberry is a different flavor and SKU.
